In [1]:
%cat /etc/lsb-release
!nvidia-smi
!nvcc --version
!python --version

DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
Tue Mar  3 21:56:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                   

In [2]:
!git clone https://github.com/anniedoris/CAD-Coder.git

fatal: destination path 'CAD-Coder' already exists and is not an empty directory.


In [3]:
%cd CAD-Coder

/content/CAD-Coder


In [4]:
#一度エラーがでてランタイムの再起動が即されますので、再起動してgit cloneから再度実施ください。２度目のインストールで成功します。
!pip uninstall -y llava transformers tokenizers sentencepiece deepspeed
!pip install -U pip
# Colab 3.12 向けに、sentencepiece だけは緩める
!pip install "transformers==4.37.2" "tokenizers==0.15.1" "sentencepiece>=0.2.0"
!pip install "accelerate==0.21.0" "peft==0.10.0" shortuuid bitsandbytes
!pip install "gradio==4.16.0" "gradio_client==0.8.1" "httpx==0.24.0"
!pip install "einops==0.6.1" "einops-exts==0.0.4" "timm==0.6.13" "scikit-learn==1.2.2"
# repo本体は依存を再解決させずに editable install
!pip install --no-deps -e .
#学習のライブラリはインストールしない
#!pip install -e ".[train]"
!pip install datasets
!pip install peft==0.10.0
!pip install tensorboard

Found existing installation: transformers 4.37.2
Uninstalling transformers-4.37.2:
  Successfully uninstalled transformers-4.37.2
Found existing installation: tokenizers 0.15.1
Uninstalling tokenizers-0.15.1:
  Successfully uninstalled tokenizers-0.15.1
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
  Using cached transformers-4.37.2-py3-none-any.whl.metadata (129 kB)
  Using cached tokenizers-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
Using cached transformers-4.37.2-py3-none-any.whl (8.4 MB)
Using cached tokenizers-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
E

In [5]:
#!pip install flash-attn --no-build-isolation
pass

## 実際のCAD QUERYでのコード作り

In [6]:
!./scripts/v1_5/eval/test_gencadcode.sh "CADCODER/CAD-Coder" "cadquery_test_data_subset100"

2026-03-03 21:57:29.315950: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772575049.338995   29652 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772575049.346554   29652 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772575049.366416   29652 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772575049.366450   29652 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772575049.366453   29652 computation_placer.cc:177] computation placer alr

### 上記で作成したコードを実行して形状をSTL,STEPで保存

In [7]:
!pip -q install cadquery trimesh plotly plyfile

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llava 1.2.2.post1 requires markdown2[all], which is not installed.
gradio 4.16.0 requires numpy~=1.0, but you have numpy 2.4.2 which is incompatible.
llava 1.2.2.post1 requires sentencepiece==0.1.99, but you have sentencepiece 0.2.1 which is incompatible.
llava 1.2.2.post1 requires torch==2.1.2, but you have torch 2.10.0+cu128 which is incompatible.
llava 1.2.2.post1 requires torchvision==0.16.2, but you have torchvision 0.25.0+cu128 which is incompatible.
esda 2.8.1 requires scikit-learn>=1.4, but you have scikit-learn 1.2.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 11.0.3 which is incompatible.
mapclassify 2.10.0 requires scikit-learn>=1.4, but you have scikit-learn 1.2.2 which 

In [8]:
!python scripts/generate_model_cad.py --dataset_name cadquery_test_data_subset100 --model_tested CADCODER/CAD-Coder --code_language cadquery --pc_reps 3 --parallel

Model name: CAD-Coder
Processing CAD tasks: 100% 100/100 [01:24<00:00,  1.18it/s]


### 正しいモデルとの比較評価

In [9]:
!python scripts/compute_iou.py --model_path CADCoder/CAD-Coder --test_set_name cadquery_test_data_subset100

  0% 0/100 [00:00<?, ?it/s]Processing: 117282.step
  1% 1/100 [00:01<01:43,  1.05s/it]Processing: 100354.step
  2% 2/100 [00:01<01:02,  1.57it/s]Processing: 9639.step
  3% 3/100 [00:02<01:09,  1.40it/s]Processing: 132441.step
  4% 4/100 [00:02<01:09,  1.38it/s]Processing: 129707.step
  5% 5/100 [00:03<00:53,  1.77it/s]Processing: 7382.step
  6% 6/100 [00:06<02:08,  1.37s/it]Processing: 93650.step
  7% 7/100 [00:06<01:35,  1.03s/it]Processing: 34378.step
  8% 8/100 [00:07<01:46,  1.16s/it]Processing: 46804.step
  9% 9/100 [00:08<01:16,  1.19it/s]Processing: 57284.step
 10% 10/100 [00:09<01:19,  1.14it/s]Processing: 104010.step
 11% 11/100 [00:09<01:06,  1.33it/s]Processing: 3872.step
 12% 12/100 [00:10<01:15,  1.17it/s]Processing: 74964.step
 13% 13/100 [00:11<01:06,  1.32it/s]Processing: 156914.step
 14% 14/100 [00:12<01:18,  1.10it/s]Processing: 124835.step
 15% 15/100 [00:13<01:23,  1.02it/s]Processing: 54522.step
 16% 16/100 [00:13<01:08,  1.23it/s]Processing: 64908.step
 17% 17/100